In [1]:
import torch
import torch.nn as nn 
import tiktoken 

In [2]:
class MultiHeadAttention(nn.Module): 
  def __init__(self, d_in, d_out, context_length, drop_rate, n_heads, qkv_bias=False): 
    super().__init__()
    self.d_in = d_in
    self.d_out = d_out
    self.context_length = context_length
    self.drop_rate = drop_rate
    self.n_heads = n_heads
    self.head_dim = d_out // n_heads 
    assert d_out % n_heads == 0 

    # weight matrices: (8, 16), PyTorch converts to (16, 8) by default
    self.W_Q = nn.Linear(d_in, d_out, bias=qkv_bias)                    
    self.W_K = nn.Linear(d_in, d_out, bias=qkv_bias)
    self.W_V = nn.Linear(d_in, d_out, bias=qkv_bias)

    # mask fill 
    self.register_buffer("mask", torch.triu(torch.ones(context_length, context_length), diagonal=1).bool())

    # dropout 
    self.dropout = nn.Dropout(drop_rate)

    # output projection 
    self.out_proj = nn.Linear(d_out, d_out)

    # kv cache 
    self.kv_cache = None

  def forward(self, x, use_cache=True):
    batch, num_tokens, d_in = x.shape 
    queries = self.W_Q(x).view(batch, num_tokens, self.n_heads, self.head_dim).transpose(2, 1)

    # kv cache 
    if self.kv_cache is None:     
      # compute q,k,v and reshape matrices
      keys = self.W_K(x).view(batch, num_tokens, self.n_heads, self.head_dim).transpose(2, 1)
      values = self.W_V(x).view(batch, num_tokens, self.n_heads, self.head_dim).transpose(2, 1)

      # update cache 
      self.kv_cache = (keys, values)
    else: 
      # compute k, v for new tokens
      queries = self.W_Q(x).view(batch, num_tokens, self.n_heads, self.head_dim).transpose(2, 1)
      keys = self.W_K(x[:,-1:,:]).view(batch, 1, self.n_heads, self.head_dim).transpose(2, 1)
      values = self.W_V(x[:,-1:,:]).view(batch, 1, self.n_heads, self.head_dim).transpose(2, 1)

      # append to cache 
      cached_keys, cached_values = self.kv_cache
      keys = torch.cat([cached_keys, keys], dim=2)
      values = torch.cat([cached_values, values], dim=2)

      # update cache 
      self.kv_cache = (keys, values)

    # attention score 
    attn_score = queries @ keys.transpose(-2, -1)
    scaled_score = attn_score / (self.head_dim ** 0.5)

    # applying causal attention with attention mask 
    scaled_score = scaled_score.masked_fill(self.mask[:num_tokens, :num_tokens], float("-inf"))

    attn_weight = torch.softmax(scaled_score, dim=-1)
    attn_weight = self.dropout(attn_weight)

    # return context vector
    context_vec = (attn_weight @ values).transpose(1,2).contiguous().view(batch, num_tokens, self.d_out)

    return self.out_proj(context_vec) 

In [3]:
class LayerNorm(nn.Module):
  def __init__(self, emb_dim):
    super().__init__()
    self.scale = nn.Parameter(torch.ones(emb_dim))
    self.shift = nn.Parameter(torch.zeros(emb_dim))
    self.eps = 1e-5

  def forward(self, x):
    mean = x.mean(dim=-1, keepdim=True)
    var = x.var(dim=-1, keepdim=True, unbiased=False) 
    norm_x = (x - mean) / torch.sqrt(var + self.eps)
    return self.scale * norm_x + self.shift 

class FeedForward(nn.Module):
  def __init__(self, emb_dim):
    super().__init__()
    self.layers = nn.Sequential(
      nn.Linear(emb_dim, 4 * emb_dim), 
      nn.GELU(approximate='tanh'),
      nn.Linear(emb_dim * 4, emb_dim), 
    )
  
  def forward(self, x):
    return self.layers(x)

In [4]:
from torch.nn import Dropout

class TransformerBlock(nn.Module):
  def __init__(self, cfg):
    super().__init__()
    self.norm1 = LayerNorm(cfg["emb_dim"])
    self.attn = MultiHeadAttention(
      d_in=cfg["emb_dim"],
      d_out=cfg["emb_dim"],
      context_length=cfg["context_length"],
      drop_rate=cfg["drop_rate"],
      n_heads=cfg["n_heads"],
      qkv_bias=cfg["qkv_bias"]
    )
    self.ff = FeedForward(cfg["emb_dim"])
    self.norm2 = LayerNorm(cfg["emb_dim"])
    self.dropout = Dropout(cfg["drop_rate"])

  def forward(self, x, use_cache=True):
    # sub-layer 1
    shortcut = x
    x = self.norm1(x)
    x = self.attn(x, use_cache=use_cache)
    x = self.dropout(x)
    x = x + shortcut

    # sub-layer 2
    shortcut = x
    x = self.norm2(x)
    x = self.ff(x)
    x = self.dropout(x)
    x = x + shortcut

    return x

In [5]:
GPT3_CONFIG_125M = {
    "vocab_size": 50257,
    "context_length": 2048,
    "emb_dim": 768,
    "n_heads": 12,
    "n_layers": 12,
    "drop_rate": 0.1,
    "qkv_bias": False,
}

In [6]:
class GPT(nn.Module):
  def __init__(self, cfg):
    super().__init__()
    self.vec_emb = nn.Embedding(cfg["vocab_size"], cfg["emb_dim"])
    self.pos_emb = nn.Embedding(cfg["context_length"], cfg["emb_dim"])
    self.dropout = nn.Dropout(cfg["drop_rate"])
    self.layers = nn.Sequential(
      # *, unpacks the list into individual arguments
      *[TransformerBlock(cfg) for _ in range(cfg["n_layers"])]
    )
    self.lnf = LayerNorm(cfg["emb_dim"])
    self.lm_head = nn.Linear(cfg["emb_dim"], cfg["vocab_size"], bias=False)

  def forward(self, x, use_cache=True):
    batch_size, seq_len = x.shape
    if use_cache and self.layers[0].attn.kv_cache is not None:
      offset = self.layers[0].attn.kv_cache[0].shape[2] 
    else: 
      offset = 0 
    pos = torch.arange(offset, offset + seq_len, device=x.device).unsqueeze(0)
    
    x = self.vec_emb(x) + self.pos_emb(pos)
    x = self.dropout(x)
    for layer in self.layers:
      x = layer(x, use_cache=use_cache)
    x = self.lnf(x)
    logits = self.lm_head(x)
    return logits



In [7]:
def reset_cache(model):
  for layer in model.layers:
    layer.attn.kv_cache = None 
  return 

def generate(model, idx, max_new_tokens, context_size):
  # reset cache from previous runs 
  reset_cache(model) 

  # prefill, generate tokens 
  logits = model(idx, use_cache=True) 
  next_token = torch.argmax(logits[:, -1, :], dim=-1, keepdim=True)  # greedy
  idx = torch.cat([idx, next_token], dim=1)   # append

  # decode, generate 1 tokens at a time 
  for _ in range(max_new_tokens - 1):
    logits = model(next_token, use_cache=True) 
    next_token = torch.argmax(logits[:, -1, :], dim=-1, keepdim=True)  # greedy
    idx = torch.cat([idx, next_token], dim=1)   # append
  return idx 

In [8]:
# Load the pretrained weights — no training in this notebook
device = torch.device("mps" if torch.backends.mps.is_available() else "cpu")
model = GPT(GPT3_CONFIG_125M)
model.load_state_dict(torch.load("gpt3_verdict.pth", map_location=device))
model.to(device)
model.eval()
print("loaded pretrained weights")

loaded pretrained weights


In [9]:
enc = tiktoken.get_encoding("gpt2")
prompt = "The sky is blue"
tokens = enc.encode(prompt)
idx = torch.tensor(tokens).unsqueeze(0).to(next(model.parameters()).device)

# to generate text
model.eval() 
with torch.no_grad():
    out = generate(model, idx, max_new_tokens=50, context_size=256)

decoded = enc.decode(out.squeeze(0).tolist())
print(decoded)

The sky is blue thought Jack Gisburn rather a cheap genius--though a good fellow enough--so it was no great surprise to me to hear that, in the height of his glory, he had dropped his painting, married a rich widow, and established himself in
